# Abnormality Detection

Detect abnormalities using healthy centroid distance.

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Abnormality Detector Class

In [ ]:
class AbnormalityDetector:
    def __init__(self, centroid_path='embeddings/mimic_healthy_centroid.pt'):
        """Initialize with healthy centroid."""
        self.centroid = torch.load(centroid_path)
        
        # Default thresholds (90th and 95th percentile from healthy distribution)
        # Adjust these based on your validation
        self.thresholds = {
            'healthy_90': 0.15,
            'healthy_95': 0.20
        }
    
    def compute_score(self, embedding):
        """
        Compute abnormality score (1 - cosine similarity with healthy centroid).
        
        Args:
            embedding: Image embedding tensor
        
        Returns:
            float: Abnormality score (0 = perfectly healthy, 1 = very abnormal)
        """
        cos_sim = torch.cosine_similarity(embedding, self.centroid, dim=-1)
        return (1.0 - cos_sim).item()
    
    def classify(self, score):
        """
        Classify abnormality level.
        
        Returns:
            str: 'normal', 'borderline', or 'abnormal'
        """
        if score < self.thresholds['healthy_90']:
            return "normal"
        elif score < self.thresholds['healthy_95']:
            return "borderline"
        else:
            return "abnormal"
    
    def analyze(self, embedding):
        """Full analysis: score + classification."""
        score = self.compute_score(embedding)
        label = self.classify(score)
        return {
            'abnormality_score': score,
            'abnormality_label': label
        }

## Initialize Detector

In [ ]:
detector = AbnormalityDetector()
print("Abnormality detector loaded")
print(f"Thresholds: {detector.thresholds}")

## Test Single Image

In [ ]:
# Load sample embedding
data = torch.load('embeddings/mimic_image_embeddings.pt')
sample_emb = data['embeddings'][0:1]

# Analyze
result = detector.analyze(sample_emb)
print(f"Abnormality Score: {result['abnormality_score']:.4f}")
print(f"Classification: {result['abnormality_label']}")

## Batch Analysis

In [ ]:
def analyze_batch(detector, embeddings):
    """Analyze multiple embeddings."""
    scores = []
    labels = []
    
    for emb in embeddings:
        result = detector.analyze(emb.unsqueeze(0))
        scores.append(result['abnormality_score'])
        labels.append(result['abnormality_label'])
    
    return scores, labels

## Visualize Distribution

In [ ]:
# Analyze a subset
subset_embeddings = data['embeddings'][:1000]
scores, labels = analyze_batch(detector, subset_embeddings)

# Plot distribution
plt.figure(figsize=(10, 6))
plt.hist(scores, bins=50, alpha=0.7, edgecolor='black')
plt.axvline(detector.thresholds['healthy_90'], color='orange', linestyle='--', label='90th percentile')
plt.axvline(detector.thresholds['healthy_95'], color='red', linestyle='--', label='95th percentile')
plt.xlabel('Abnormality Score')
plt.ylabel('Frequency')
plt.title('Abnormality Score Distribution')
plt.legend()
plt.show()

# Count by category
from collections import Counter
label_counts = Counter(labels)
print("\nDistribution:")
for label, count in label_counts.items():
    print(f"  {label}: {count} ({count/len(labels)*100:.1f}%)")

## Load Pre-computed Scores (if available)

In [ ]:
# If you have pre-computed scores
# scores_data = torch.load('embeddings/mimic_abnormality_scores.pt')
# print(f"Loaded {len(scores_data)} pre-computed scores")